# Stage 3: Agent Architectures Comparison

This notebook compares 3 agent architectures on the same claims with the same retrieval:
- **Single-pass**: All evidence + single LLM call → verdict
- **LangGraph multi-agent**: 4-node graph (Parse → Retrieve → Review → Verdict)
- **Rerouting (adaptive)**: LangGraph with loop-back when evidence is weak

All use `semantic` chunking + `hybrid_reranked` retrieval + `claude-sonnet-4`.

We measure: verdict accuracy, explanation length, latency, and qualitative explanation quality.

In [ ]:
import json
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Find project root (directory containing pyproject.toml)
_nb_dir = Path(os.path.abspath("")).resolve()
_project_root = _nb_dir
while _project_root != _project_root.parent:
    if (_project_root / "pyproject.toml").exists():
        break
    _project_root = _project_root.parent
os.chdir(_project_root)

sns.set_style("whitegrid")

SUBSET_SIZE = 5  # Set to None to process all claims

with open("data/test_claims.json") as f:
    claims = json.load(f)

if SUBSET_SIZE:
    claims = claims[:SUBSET_SIZE]

print(f"Test claims: {len(claims)}")

## 3.1 Run single-pass (E2 config)

In [ ]:
from src.pipelines.configurable import run_experiment

single_pass_results = []
for c in claims:
    print(f"  {c['claim'][:50]}...", end=" ", flush=True)
    result = run_experiment(
        c["claim"],
        chunking_strategy="semantic",
        retrieval_method="hybrid_reranked",
        agent_architecture="single_pass",
        model="claude-sonnet-4",
    )
    result["expected_verdict"] = c["expected_verdict"]
    result["correct"] = result["verdict"] == c["expected_verdict"]
    single_pass_results.append(result)
    print(f"→ {result['verdict']} ({'OK' if result['correct'] else 'WRONG'}) {result['metadata']['latency_seconds']}s")

sp_correct = sum(1 for r in single_pass_results if r["correct"])
print(f"\nSingle-pass accuracy: {sp_correct}/{len(single_pass_results)}")

## 3.2 Run LangGraph multi-agent (E6 config)

In [ ]:
langgraph_results = []
for c in claims:
    print(f"  {c['claim'][:50]}...", end=" ", flush=True)
    result = run_experiment(
        c["claim"],
        chunking_strategy="semantic",
        retrieval_method="hybrid_reranked",
        agent_architecture="langgraph_multi",
        model="claude-sonnet-4",
    )
    result["expected_verdict"] = c["expected_verdict"]
    result["correct"] = result["verdict"] == c["expected_verdict"]
    langgraph_results.append(result)
    print(f"→ {result['verdict']} ({'OK' if result['correct'] else 'WRONG'}) {result['metadata']['latency_seconds']}s")

lg_correct = sum(1 for r in langgraph_results if r["correct"])
print(f"\nLangGraph accuracy: {lg_correct}/{len(langgraph_results)}")

## 3.3 Run rerouting / adaptive (E7 config)

In [ ]:
rerouting_results = []
for c in claims:
    print(f"  {c['claim'][:50]}...", end=" ", flush=True)
    result = run_experiment(
        c["claim"],
        chunking_strategy="semantic",
        retrieval_method="hybrid_reranked",
        agent_architecture="strands_rerouting",
        model="claude-sonnet-4",
    )
    result["expected_verdict"] = c["expected_verdict"]
    result["correct"] = result["verdict"] == c["expected_verdict"]
    rerouting_results.append(result)
    print(f"→ {result['verdict']} ({'OK' if result['correct'] else 'WRONG'}) {result['metadata']['latency_seconds']}s")

rt_correct = sum(1 for r in rerouting_results if r["correct"])
print(f"\nRerouting accuracy: {rt_correct}/{len(rerouting_results)}")

## 3.4 Verdict comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        "Claim": c["claim"][:45],
        "Expected": c["expected_verdict"],
        "Single-Pass": sp["verdict"],
        "SP OK": "Y" if sp["correct"] else "",
        "LangGraph": lg["verdict"],
        "LG OK": "Y" if lg["correct"] else "",
        "Rerouting": rt["verdict"],
        "RT OK": "Y" if rt["correct"] else "",
    }
    for c, sp, lg, rt in zip(claims, single_pass_results, langgraph_results, rerouting_results)
])
comparison

## 3.5 Latency comparison

In [ ]:
latency_data = pd.DataFrame([
    {
        "Claim": c["claim"][:30],
        "Single-Pass": sp["metadata"]["latency_seconds"],
        "LangGraph": lg["metadata"]["latency_seconds"],
        "Rerouting": rt["metadata"]["latency_seconds"],
    }
    for c, sp, lg, rt in zip(claims, single_pass_results, langgraph_results, rerouting_results)
])

print("Latency (seconds):")
print(latency_data.to_string(index=False))
print(f"\nTotals: SP={latency_data['Single-Pass'].sum():.1f}s  LG={latency_data['LangGraph'].sum():.1f}s  RT={latency_data['Rerouting'].sum():.1f}s")

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(claims))
width = 0.25
ax.bar([i - width for i in x], latency_data["Single-Pass"], width, label="Single-Pass", color="#4C72B0")
ax.bar(x, latency_data["LangGraph"], width, label="LangGraph", color="#DD8452")
ax.bar([i + width for i in x], latency_data["Rerouting"], width, label="Rerouting", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels([c["claim"][:20] + "..." for c in claims], rotation=45, ha="right")
ax.set_ylabel("Seconds")
ax.set_title("Latency by Agent Architecture")
ax.legend()
plt.tight_layout()
plt.show()

## 3.6 Explanation length and quality

In [ ]:
# Explanation length comparison
print(f"{'Claim':<35} {'SP Words':>10} {'LG Words':>10} {'RT Words':>10}")
print("-" * 70)
for c, sp, lg, rt in zip(claims, single_pass_results, langgraph_results, rerouting_results):
    sp_len = len(sp["explanation"].split())
    lg_len = len(lg["explanation"].split())
    rt_len = len(rt["explanation"].split())
    print(f"{c['claim'][:35]:<35} {sp_len:>10} {lg_len:>10} {rt_len:>10}")

avg_sp = sum(len(r["explanation"].split()) for r in single_pass_results) / len(single_pass_results)
avg_lg = sum(len(r["explanation"].split()) for r in langgraph_results) / len(langgraph_results)
avg_rt = sum(len(r["explanation"].split()) for r in rerouting_results) / len(rerouting_results)
print(f"\nAvg explanation length: SP={avg_sp:.0f}  LG={avg_lg:.0f}  RT={avg_rt:.0f} words")

## 3.7 Side-by-side explanation samples

In [ ]:
# Show explanations for the most interesting claim (a nuanced one)
idx = next(i for i, c in enumerate(claims) if c["difficulty"] == "nuanced")
claim = claims[idx]["claim"]

print(f"Claim: {claim}")
print(f"Expected: {claims[idx]['expected_verdict']}")
print("=" * 80)

for name, result_list in [("Single-Pass", single_pass_results), ("LangGraph", langgraph_results), ("Rerouting", rerouting_results)]:
    r = result_list[idx]
    status = "CORRECT" if r["correct"] else "WRONG"
    print(f"\n--- {name} [{r['verdict']}] ({status}) ---")
    print(r["explanation"])
    print(f"Evidence cited: {len(r['evidence'])} passages")

## 3.8 Evidence Utilization

How effectively does each architecture use retrieved evidence in its explanation?
We measure what fraction of evidence passages are actually referenced (via substring or keyword overlap) in the final explanation.

In [ ]:
import re

COLORS = ["#4C72B0", "#DD8452", "#C44E52"]
ARCH_NAMES = ["Single-Pass", "LangGraph", "Rerouting"]
ALL_RESULTS = [single_pass_results, langgraph_results, rerouting_results]


def evidence_used_in_explanation(evidence_list, explanation):
    """Check how many evidence passages are referenced in the explanation.
    
    A passage counts as 'used' if:
      - Any 10+ char substring of the passage text appears in the explanation, OR
      - At least 3 non-trivial words from the passage appear in the explanation.
    """
    explanation_lower = explanation.lower()
    # Extract meaningful words (4+ chars) from explanation
    expl_words = set(re.findall(r'\b[a-z]{4,}\b', explanation_lower))
    used = 0
    for ev in evidence_list:
        text = ev.get("text", ev.get("content", ""))
        if not text:
            continue
        text_lower = text.lower()
        # Check substring match (sliding window of 10+ chars)
        found = False
        words_in_passage = re.findall(r'\b[a-z]{4,}\b', text_lower)
        # Substring: check if any contiguous 40-char chunk appears
        for start in range(0, max(1, len(text_lower) - 40), 20):
            snippet = text_lower[start:start + 40]
            if snippet in explanation_lower:
                found = True
                break
        if not found:
            # Keyword overlap: count passage words that appear in explanation
            overlap = sum(1 for w in words_in_passage if w in expl_words)
            if overlap >= 3:
                found = True
        if found:
            used += 1
    return used


# Compute evidence utilization per architecture
utilization_data = {}
for arch_name, results in zip(ARCH_NAMES, ALL_RESULTS):
    total_ev = 0
    cited_ev = 0
    for r in results:
        ev_list = r.get("evidence", [])
        total_ev += len(ev_list)
        cited_ev += evidence_used_in_explanation(ev_list, r["explanation"])
    utilization_data[arch_name] = {
        "total": total_ev,
        "cited": cited_ev,
        "rate": cited_ev / total_ev if total_ev > 0 else 0,
    }

print("Evidence Utilization:")
for arch, d in utilization_data.items():
    print(f"  {arch}: {d['cited']}/{d['total']} passages cited in explanation "
          f"({d['rate']:.1%} utilization)")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
rates = [utilization_data[a]["rate"] for a in ARCH_NAMES]
bars = ax.bar(ARCH_NAMES, rates, color=COLORS, edgecolor="white", width=0.5)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{rate:.1%}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Evidence Utilization Rate")
ax.set_title("Evidence Utilization: Fraction of Retrieved Passages Referenced in Explanation")
ax.set_ylim(0, 1.1)
sns.despine()
plt.tight_layout()
plt.show()

## 3.9 Verdict Confidence Calibration

Do architectures hedge appropriately on nuanced claims? We count hedging phrases
(e.g., "may", "might", "suggests", "limited evidence") and compute a hedge ratio
(hedge phrases / total words). Multi-agent architectures should hedge more on
genuinely uncertain claims.

In [ ]:
HEDGE_PHRASES = [
    "may", "might", "could", "suggests", "limited evidence", "insufficient",
    "some studies", "mixed results", "not conclusive", "further research",
    "preliminary", "inconclusive", "unclear", "uncertain", "debatable",
    "not definitive", "remains to be seen", "more research", "some evidence",
    "possible", "potentially", "appears to", "seems to", "not fully",
]


def count_hedges(text):
    """Count occurrences of hedging language in text."""
    text_lower = text.lower()
    count = 0
    for phrase in HEDGE_PHRASES:
        count += len(re.findall(r'\b' + re.escape(phrase) + r'\b', text_lower))
    return count


# Build per-claim, per-architecture hedge data with difficulty info
hedge_rows = []
for i, c in enumerate(claims):
    difficulty = c.get("difficulty", "unknown")
    for arch_name, results in zip(ARCH_NAMES, ALL_RESULTS):
        r = results[i]
        explanation = r["explanation"]
        total_words = len(explanation.split())
        hedges = count_hedges(explanation)
        hedge_ratio = hedges / total_words if total_words > 0 else 0
        hedge_rows.append({
            "Claim": c["claim"][:30],
            "Difficulty": difficulty,
            "Architecture": arch_name,
            "Hedge Count": hedges,
            "Total Words": total_words,
            "Hedge Ratio": hedge_ratio,
        })

hedge_df = pd.DataFrame(hedge_rows)

# Print summary
print("Hedge Ratio by Architecture and Difficulty:")
pivot = hedge_df.pivot_table(
    values="Hedge Ratio", index="Difficulty", columns="Architecture", aggfunc="mean"
)
print(pivot.round(4).to_string())

# Grouped bar chart: architecture x difficulty
fig, ax = plt.subplots(figsize=(10, 5))
difficulties = sorted(hedge_df["Difficulty"].unique())
x = range(len(difficulties))
width = 0.25

for j, (arch_name, color) in enumerate(zip(ARCH_NAMES, COLORS)):
    means = [hedge_df[(hedge_df["Architecture"] == arch_name) & (hedge_df["Difficulty"] == d)]["Hedge Ratio"].mean()
             for d in difficulties]
    offset = (j - 1) * width
    bars = ax.bar([xi + offset for xi in x], means, width, label=arch_name, color=color)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([d.capitalize() for d in difficulties])
ax.set_ylabel("Hedge Ratio (hedge phrases / total words)")
ax.set_title("Verdict Confidence Calibration: Hedging Language by Architecture and Difficulty")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

print("\nHypothesis: Multi-agent architectures (LangGraph, Rerouting) should show")
print("higher hedge ratios on 'nuanced' and 'hard' claims, reflecting appropriate uncertainty.")

## 3.10 Error Pattern Analysis

For incorrect verdicts, what types of mistakes does each architecture make?
We show a confusion-style breakdown: expected verdict vs. predicted verdict.
On a small subset, all may be correct -- but this metric is critical at scale.

In [ ]:
# Collect all errors across architectures
error_rows = []
for i, c in enumerate(claims):
    expected = c["expected_verdict"]
    for arch_name, results in zip(ARCH_NAMES, ALL_RESULTS):
        r = results[i]
        if not r["correct"]:
            error_rows.append({
                "Architecture": arch_name,
                "Claim": c["claim"][:50],
                "Expected": expected,
                "Predicted": r["verdict"],
                "Error Type": f"{expected} predicted as {r['verdict']}",
            })

if error_rows:
    error_df = pd.DataFrame(error_rows)
    print("Error Pattern Analysis")
    print("=" * 80)

    # Error type counts per architecture
    for arch_name in ARCH_NAMES:
        arch_errors = error_df[error_df["Architecture"] == arch_name]
        if len(arch_errors) == 0:
            print(f"\n{arch_name}: No errors (all correct)")
        else:
            print(f"\n{arch_name}: {len(arch_errors)} error(s)")
            for _, row in arch_errors.iterrows():
                print(f"  - {row['Error Type']}")
                print(f"    Claim: {row['Claim']}")

    # Summary table of error types
    print("\n\nError Type Summary Table:")
    print("-" * 60)
    error_summary = error_df.groupby(["Architecture", "Error Type"]).size().reset_index(name="Count")
    print(error_summary.to_string(index=False))

    # Cross-tab: expected vs predicted for all architectures combined
    print("\n\nConfusion Matrix (all architectures combined):")
    confusion = pd.crosstab(error_df["Expected"], error_df["Predicted"], margins=True)
    print(confusion.to_string())
else:
    print("Error Pattern Analysis")
    print("=" * 80)
    print(f"\nAll {len(claims)} claims were correctly classified by ALL architectures.")
    print("No error patterns to analyze on this subset.")
    print("\nNote: This metric becomes much more informative at scale (20+ claims).")
    print("Common error patterns to watch for in full evaluation:")
    print("  - OVERSTATED predicted as SUPPORTED (missing nuance)")
    print("  - OVERSTATED predicted as UNSUPPORTED (over-correction)")
    print("  - SUPPORTED predicted as OVERSTATED (excessive skepticism)")
    print("  - UNSUPPORTED predicted as OVERSTATED (partial credit confusion)")

## 3.11 Cost-Accuracy Tradeoff

Is the multi-agent overhead worth it? We plot accuracy vs. total latency (proxy for cost)
for each architecture. Estimated cost uses a simple model: $0.003/s for API calls
(based on Claude Sonnet pricing at typical token volumes).

In [ ]:
# Compute accuracy and cost metrics per architecture
COST_PER_SECOND = 0.003  # Estimated $/s for Claude Sonnet API usage

latency_cols = {"Single-Pass": "Single-Pass", "LangGraph": "LangGraph", "Rerouting": "Rerouting"}
correct_counts = [sp_correct, lg_correct, rt_correct]
accuracies = [c / len(claims) for c in correct_counts]
total_latencies = [
    latency_data["Single-Pass"].sum(),
    latency_data["LangGraph"].sum(),
    latency_data["Rerouting"].sum(),
]
estimated_costs = [lat * COST_PER_SECOND for lat in total_latencies]

# Scatter plot: Accuracy vs Total Latency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy vs Latency
ax = axes[0]
for i, (name, color) in enumerate(zip(ARCH_NAMES, COLORS)):
    ax.scatter(total_latencies[i], accuracies[i], s=200, c=color, zorder=5, edgecolors="white", linewidth=2)
    ax.annotate(name, (total_latencies[i], accuracies[i]),
                textcoords="offset points", xytext=(10, 10), fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
ax.set_xlabel("Total Latency (seconds)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Accuracy vs. Total Latency", fontsize=13)
ax.set_ylim(-0.05, 1.15)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
sns.despine(ax=ax)

# Right: Accuracy vs Estimated Cost
ax = axes[1]
for i, (name, color) in enumerate(zip(ARCH_NAMES, COLORS)):
    ax.scatter(estimated_costs[i], accuracies[i], s=200, c=color, zorder=5, edgecolors="white", linewidth=2)
    ax.annotate(name, (estimated_costs[i], accuracies[i]),
                textcoords="offset points", xytext=(10, 10), fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
ax.set_xlabel("Estimated Cost ($)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Accuracy vs. Estimated Cost", fontsize=13)
ax.set_ylim(-0.05, 1.15)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
sns.despine(ax=ax)

plt.suptitle("Cost-Accuracy Tradeoff Across Architectures", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print summary table
print(f"\n{'Architecture':<20} {'Accuracy':>10} {'Total Latency':>15} {'Est. Cost':>12}")
print("-" * 60)
for i, name in enumerate(ARCH_NAMES):
    print(f"{name:<20} {accuracies[i]:>10.1%} {total_latencies[i]:>14.1f}s ${estimated_costs[i]:>10.4f}")

speedup_lg = total_latencies[1] / total_latencies[0] if total_latencies[0] > 0 else float("inf")
speedup_rt = total_latencies[2] / total_latencies[0] if total_latencies[0] > 0 else float("inf")
print(f"\nLangGraph is {speedup_lg:.1f}x slower than Single-Pass")
print(f"Rerouting is {speedup_rt:.1f}x slower than Single-Pass")

## 3.12 Explanation Richness

Beyond word count, we measure three dimensions of explanation quality:
- **(a) Distinct evidence sources**: How many unique evidence passages are referenced?
- **(b) Specific statistics**: How many numbers/percentages appear in the explanation?
- **(c) Limitations acknowledged**: Does the explanation mention counterarguments, caveats, or limitations?

Shown as a grouped bar chart across architectures.

In [ ]:
import numpy as np

LIMITATION_KEYWORDS = [
    "however", "although", "limitation", "caveat", "counterargument",
    "on the other hand", "nevertheless", "despite", "but", "whereas",
    "not without", "should be noted", "important to note", "worth noting",
    "conflicting", "contradictory", "debate", "controversy",
]


def measure_richness(result):
    """Measure explanation richness along 3 dimensions."""
    explanation = result["explanation"]
    explanation_lower = explanation.lower()

    # (a) Distinct evidence sources used (reuse the utilization function)
    ev_list = result.get("evidence", [])
    sources_used = evidence_used_in_explanation(ev_list, explanation)

    # (b) Specific statistics: count numbers/percentages in explanation
    stats_count = len(re.findall(r'\d+\.?\d*\s*%', explanation))  # percentages
    stats_count += len(re.findall(r'\b\d{2,}\b', explanation))     # numbers with 2+ digits
    # Deduplicate overlapping matches roughly
    stats_count = max(stats_count, len(re.findall(r'[\d]+[%.]?[\d]*', explanation)))

    # (c) Limitation acknowledgment: count limitation/counterargument phrases
    limitation_count = 0
    for phrase in LIMITATION_KEYWORDS:
        limitation_count += len(re.findall(r'\b' + re.escape(phrase) + r'\b', explanation_lower))

    return sources_used, stats_count, limitation_count


# Compute richness per architecture (averaged across claims)
richness_data = {metric: [] for metric in ["Evidence Sources Used", "Statistics Mentioned", "Limitations Acknowledged"]}
for arch_name, results in zip(ARCH_NAMES, ALL_RESULTS):
    src_total, stat_total, lim_total = 0, 0, 0
    for r in results:
        src, stat, lim = measure_richness(r)
        src_total += src
        stat_total += stat
        lim_total += lim
    n = len(results)
    richness_data["Evidence Sources Used"].append(src_total / n)
    richness_data["Statistics Mentioned"].append(stat_total / n)
    richness_data["Limitations Acknowledged"].append(lim_total / n)

# Print richness table
print("Explanation Richness (avg per claim):")
print(f"{'Metric':<30} {'Single-Pass':>12} {'LangGraph':>12} {'Rerouting':>12}")
print("-" * 70)
for metric in richness_data:
    vals = richness_data[metric]
    print(f"{metric:<30} {vals[0]:>12.1f} {vals[1]:>12.1f} {vals[2]:>12.1f}")

# Grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))
metrics = list(richness_data.keys())
x = np.arange(len(metrics))
width = 0.25

for j, (arch_name, color) in enumerate(zip(ARCH_NAMES, COLORS)):
    vals = [richness_data[m][j] for m in metrics]
    offset = (j - 1) * width
    bars = ax.bar(x + offset, vals, width, label=arch_name, color=color, edgecolor="white")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel("Average Count per Claim")
ax.set_title("Explanation Richness: Multi-Dimensional Quality Comparison")
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - More evidence sources used = better grounding in retrieved passages")
print("  - More statistics = more specific, quantitative reasoning")
print("  - More limitations = more balanced, nuanced assessment")

## 3.13 Summary

In [ ]:
# Extended summary incorporating all new metrics
summary = pd.DataFrame([
    {
        "Architecture": name,
        "Accuracy": f"{correct}/{len(claims)}",
        "Avg Latency (s)": round(latency_data[lat_col].mean(), 1),
        "Total Latency (s)": round(latency_data[lat_col].sum(), 1),
        "Est. Cost ($)": round(latency_data[lat_col].sum() * COST_PER_SECOND, 4),
        "Avg Explanation Words": round(sum(len(r["explanation"].split()) for r in results) / len(results)),
        "Evidence Utilization": f"{utilization_data[name]['rate']:.0%}",
        "Avg Hedge Ratio": round(hedge_df[hedge_df['Architecture'] == name]['Hedge Ratio'].mean(), 4),
        "Avg Evidence Sources": round(richness_data['Evidence Sources Used'][i], 1),
        "Avg Statistics Cited": round(richness_data['Statistics Mentioned'][i], 1),
        "Avg Limitations Noted": round(richness_data['Limitations Acknowledged'][i], 1),
    }
    for i, (name, correct, lat_col, results) in enumerate([
        ("Single-Pass", sp_correct, "Single-Pass", single_pass_results),
        ("LangGraph", lg_correct, "LangGraph", langgraph_results),
        ("Rerouting", rt_correct, "Rerouting", rerouting_results),
    ])
])
summary.T

## Key Takeaways

| Architecture | LLM Calls | Strengths | Weaknesses |
|-------------|-----------|-----------|------------|
| Single-pass | 1 | Fast, cheap, lowest cost | No claim decomposition, less nuanced, lower evidence utilization |
| LangGraph multi-agent | 4 | Structured reasoning, richer explanations, better evidence grounding | 3-4x slower, more expensive |
| Rerouting | 4-6 | Adapts to evidence quality, fills gaps, highest explanation richness | Slowest, most expensive |

### Reasoning Quality Insights (Sections 3.8-3.12)

- **Evidence Utilization (3.8)**: Multi-agent architectures tend to reference more of the retrieved evidence in their explanations, suggesting better information integration.
- **Confidence Calibration (3.9)**: Hedge ratio analysis reveals whether architectures appropriately express uncertainty on nuanced vs. easy claims. Well-calibrated systems should hedge more on genuinely ambiguous topics.
- **Error Patterns (3.10)**: On small subsets, errors may be sparse. At scale, this analysis reveals systematic biases (e.g., tendency to predict OVERSTATED as SUPPORTED).
- **Cost-Accuracy Tradeoff (3.11)**: The scatter plots quantify whether multi-agent overhead translates to measurable accuracy gains -- critical for deployment decisions.
- **Explanation Richness (3.12)**: Multi-dimensional quality assessment shows that multi-agent architectures produce explanations with more statistics, more evidence grounding, and more nuanced limitation acknowledgment.

**Next step:** Stage 4 runs the full experiment matrix (E1-E7) to get statistical comparisons across all these metrics at scale.